<a href="https://colab.research.google.com/github/zegmundo-koziel/Projeto-Avaliativo-M1-S14/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## --- Import das bibliotecas e dataframe ---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar o arquivo CSV
df = pd.read_csv('credit_risk_dataset.csv')


## --- FASE 1: Análise Exploratória de Dados (EDA) ---

#  Analise Descritiva e Estatística

In [ ]:
# Exibir as primeiras 5 linhas para verificar se os dados carregaram corretamente
print("--- Primeiras Linhas do Dataset ---")
#print(df.head())
display(df.head())

### 📋 Dicionário de Dados Simplificado

* **`person_age`**: Idade do cliente.
* **`person_income`**: Renda anual do cliente.
* **`person_home_ownership`**: Situação da moradia (se é alugada, própria ou hipotecada).
* **`person_emp_length`**: Tempo (em anos) que a pessoa está no emprego atual.
* **`loan_intent`**: Motivo ou intenção do empréstimo (ex: pessoal, educação, médico).
* **`loan_grade`**: Nota/Classe do empréstimo (grau de risco que o banco deu para o cliente).
* **`loan_amnt`**: Valor total do empréstimo solicitado.
* **`loan_int_rate`**: Taxa de juros do empréstimo.
* **`loan_status`**: Status do empréstimo (0 = Pago em dia / 1 = Deixou de pagar). `Esta é a variável alvo.`
* **`loan_percent_income`**: Quanto o valor do empréstimo representa do total da renda anual do cliente (em percentual).
* **`cb_person_default_on_file`**: Indica se o cliente já tem histórico de inadimplência ou restrição no nome.
* **`cb_person_cred_hist_length`**: Tempo (em anos) desde que o cliente abriu sua primeira linha de crédito no mercado.

In [ ]:
# Exibir informações sobre a quantidade de colunas e linhas do dataframe
print(f"O dataframe possui {df.shape[1]} colunas e {df.shape[0]} linhas.")

# Exibir informações gerais sobre o tipo das colunas e dados nulos
print("\n--- Informações Gerais do Dataset ---")
print(df.info())

In [ ]:
# Exibir informações de valores nulos por coluna
print ("---Soma dos valores nulos por coluna---")
df.isnull().sum()

In [ ]:
# Exibir o resumo estatístico do dataframe # Média, mediana, desvio padrão e outros 
print ("---Resumo Estatístico Descritivo - Numéricas---")
display(df.describe())

print("\n---Sumário Estatístico Descritivo - Categorias---")
display(df.describe(include=['object', 'str']))

In [ ]:
# Exibir a quantidade de linhas duplicadas 
print("---Total de linhas duplicadas encontrado---")
print(f"Encontrado {df.duplicated().sum()} linhas duplicadas.")

# exibir as linhas duplicadas na tela
#print(df[df.duplicated()])


# Analise visual - gráficos

In [ ]:
# Gráfico 1: Histograma de Distribuição de Idades - person_age
plt.figure(figsize=(10, 5))
sns.histplot(df['person_age'], bins=40, kde=True, color='royalblue')
plt.title('Distribuição da Idade dos Clientes (person_age)', fontsize=14, pad=15)
plt.xlabel('Idade (Anos)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.xlim(18, 80) # Corta idades extremas/outliers comuns nessa base para melhor visualização
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Desbalanceamento da Variável Alvo - loan_status

plt.figure(figsize=(6, 5))
ax = sns.countplot(x='loan_status', data=df, hue='loan_status', palette='Set2', legend=False)
plt.title('Desbalanceamento da Variável Alvo (loan_status)', fontsize=14, pad=15)
plt.xlabel('Status do Empréstimo (0 = Pago, 1 = Inadimplente)', fontsize=12)
plt.ylabel('Quantidade de Registros', fontsize=12)

total_registros = len(df)
for container in ax.containers:
    porcentagem = [f'{valor}\n({(valor / total_registros) * 100:.1f}%)' for valor in container.datavalues]
    ax.bar_label(container, labels=porcentagem, fontsize=10, label_type='center', color='black')

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 3: Mapa de Calor de Correlação de Pearson
plt.figure(figsize=(10, 8))
# Seleciona apenas as colunas numéricas da sua lista
colunas_numericas = df.select_dtypes(include=['int64', 'float64'])
matriz_correlacao = colunas_numericas.corr(method='pearson')

sns.heatmap(
    matriz_correlacao, 
    annot=True, 
    cmap='coolwarm', 
    fmt='.2f', 
    linewidths=0.5, 
    square=True, 
    cbar_kws={"shrink": .8},
    vmin=-1, vmax=1
)

plt.title('Mapa de Calor - Correlação de Pearson', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 4: Gráfico de Dispersão para Identificação de Outliers
plt.figure(figsize=(10, 6))

# Idade vs Tempo de Emprego e colorindo pelo Status do Empréstimo
sns.scatterplot(
    data=df, 
    x='person_age', 
    y='person_emp_length', 
    hue='loan_status', 
    palette='Set1', 
    alpha=0.7, 
    edgecolor='w'
)

plt.title('Gráfico de Dispersão: Idade vs. Tempo de Emprego (Identificação de Outliers)', fontsize=14, pad=15)
plt.xlabel('Idade do Cliente (person_age)', fontsize=12)
plt.ylabel('Tempo de Emprego em Anos (person_emp_length)', fontsize=12)
plt.legend(title='Status do Empréstimo\n(0 = Pago, 1 = Inadimplente)', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


# Tomada de Decisão:

Tomada de Decisão e Estratégia 

A análise estatística descritiva e os quatro gráficos gerados revelaram problemas críticos que guiarão nossa estratégia de preparação na próxima fase, evitando o problema do *Garbage In, Garbage Out*:

1. **Tratamento de Outliers (Provado pelo Gráfico 1 e 4):** O gráfico de dispersão e o histograma deixam claro que existem erros gritantes de digitação na base, como pessoas com mais de 100 anos de idade (`person_age`) e tempos de emprego que ultrapassam 120 anos (`person_emp_length`). Na Fase 2, aplicaremos filtros lógicos para eliminar esses registros impossíveis (ex: manter apenas `person_age < 80` e `person_emp_length < 50`), limpando o ruído que prejudicaria os cálculos de distância do algoritmo KNN.
2. **Correção de Dados Ausentes e Duplicados:** A função `.isnull().sum()` e a checagem de duplicados acusaram a presença de dados nulos nas colunas de tempo de emprego e taxa de juros (`loan_int_rate`), além de linhas duplicadas. Removeremos as duplicadas para não gerar redundância. Como estratégia mista para dados ausentes, eliminaremos completamente as linhas com nulos em `person_emp_length` (por representar apenas 3% da base e indicar possível inconsistência cadastral) e faremos a imputação por **mediana** apenas na coluna `loan_int_rate` (10% da base), blindando nosso volume de dados contra perdas massivas e evitando distorções causadas pela assimetria.
3. **Estratégia para o Desbalanceamento (Provado pelo Gráfico 2):** O gráfico de barras confirmou que a classe majoritária (0 - Pago) representa a esmagadora maioria dos dados (aproximadamente 78%). Se ignorarmos isso, nossos modelos falharão criticamente na detecção de inadimplentes. Para resolver o problema, utilizaremos técnicas de reamostragem (como SMOTE) ou ajustaremos os pesos das classes (`class_weight='balanced'`) diretamente nos modelos KNN e Árvore de Decisão.
4. **Codificação e Escalonamento (Provado pelo Gráfico 3):** O mapa de calor de Pearson indicou baixas correlações lineares diretas com o alvo, o que justifica o uso de modelos não-lineares como as Árvores de Decisão. Para o KNN, variáveis categóricas serão transformadas via *One-Hot Encoding* e aplicaremos uma padronização rigorosa de escala (`StandardScaler`) em todas as variáveis numéricas, garantindo que o valor da renda anual (`person_income`) não soterre o peso das outras colunas no cálculo de proximidade do modelo.



## --- Fase 2: Tratamento e Limpeza (Data Prep) ---

# Identificação e Remoção de Duplicadas

In [ ]:
print("--- Iniciando o Pipeline de Preparação de Dados ---\n")

total_duplicadas = df.duplicated().sum()
df_tratado = df.drop_duplicates().copy()
print(f" Linhas duplicadas removidas: {total_duplicadas}")
# Exibe o tamanho do dataframe após a remoção de duplicadas
print(f"O dataframe possui {df_tratado.shape[1]} colunas e {df_tratado.shape[0]} linhas.")

# Tratamento de Valores Nulos

In [ ]:
print("\nValores nulos antes da imputação: person_emp_length")
print(df_tratado['person_emp_length'].isnull().sum())
# Remoção de Nulos da Coluna 'person_emp_length'
linhas_nulas = df_tratado.shape[0]
df_tratado = df_tratado.dropna(subset=['person_emp_length'])
linhas_nulas_removidas = linhas_nulas - df_tratado.shape[0]
print(f"Linhas nulas removidas em 'person_emp_length': {linhas_nulas_removidas} linhas")

# Imputação pela Mediana na Coluna 'loan_int_rate'
print("\nValores nulos antes da imputação: loan_int_rate")
print(df_tratado['loan_int_rate'].isnull().sum())
mediana_taxa_juros = df_tratado['loan_int_rate'].median()
print(f"Mediana da coluna 'loan_int_rate': {mediana_taxa_juros}")
df_tratado['loan_int_rate'] = df_tratado['loan_int_rate'].fillna(mediana_taxa_juros)
print(f"Valores nulos em 'loan_int_rate' após tratamento com Mediana: {df_tratado['loan_int_rate'].isnull().sum()}")
print(f"\nTamanho final do dataset limpo: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas.")

# Tratamento de Outliers

In [ ]:
linhas_antes_outliers = df_tratado.shape
df_tratado = df_tratado[df_tratado['person_age'] < 80]
df_tratado = df_tratado[df_tratado['person_emp_length'] < 50] 
linhas_removidas_outliers = linhas_antes_outliers[0] - df_tratado.shape[0]
print(f"Outliers extremos removidos (Idade > 80 ou Emprego > 50): {linhas_removidas_outliers} linhas")


In [ ]:
# diferença de linhas e colunas entre o dataset original e o dataset tratado
total_linhas_removidas = df.shape[0] - df_tratado.shape[0]
porcentagem_linhas_removidas = (total_linhas_removidas / df.shape[0]) * 100
# Linhas tratadas com a mediana
linhas_tratadas_mediana = df['loan_int_rate'].isnull().sum()
porcentagem_linhas_tratadas = (linhas_tratadas_mediana / df.shape[0]) * 100

print("--- RESUMO DO TRATAMENTO DE DADOS ---")
print(f"- Total de linhas removidas durante o tratamento: {total_linhas_removidas} linhas")   
print(f"- Porcentagem de linhas removidas: {porcentagem_linhas_removidas:.2f}%")   
print(f"- Linhas tratadas com mediana: {linhas_tratadas_mediana} linhas") 
print(f"- Porcentagem de linhas tratadas com mediana: {porcentagem_linhas_tratadas:.2f}%")


In [ ]:
# Gráfico 5: Gráfico de Dispersão de Outliers pós-tratamento
plt.figure(figsize=(10, 6))

# Idade vs Tempo de Emprego e colorindo pelo Status do Empréstimo
sns.scatterplot(
    data=df_tratado, 
    x='person_age', 
    y='person_emp_length', 
    hue='loan_status', 
    palette='Set1', 
    alpha=0.7, 
    edgecolor='w'
)

plt.title('Gráfico de Dispersão: Idade vs. Tempo de Emprego (Pós-Tratamento de Outliers)', fontsize=14, pad=15)
plt.xlabel('Idade do Cliente (person_age)', fontsize=12)
plt.ylabel('Tempo de Emprego em Anos (person_emp_length)', fontsize=12)
plt.legend(title='Status do Empréstimo\n(0 = Pago, 1 = Inadimplente)', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 6: Boxplots para Validação de Outliers Pós-Tratamento
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot 1: Idade do Cliente por Status do Empréstimo
sns.boxplot(
    data=df_tratado, 
    x='loan_status', 
    y='person_age', 
    ax=axes[0],               
    hue='loan_status',        
    palette='Set2', 
    legend=False              
)
axes[0].set_title('Distribuição da Idade (person_age)', fontsize=12, pad=10)
axes[0].set_xlabel('Status do Empréstimo (0=Pago, 1=Inadimplente)', fontsize=10)
axes[0].set_ylabel('Idade em Anos', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.3)

# Boxplot 2: Tempo de Emprego por Status do Empréstimo
sns.boxplot(
    data=df_tratado, 
    x='loan_status', 
    y='person_emp_length', 
    ax=axes[1],               
    hue='loan_status',        
    palette='Set2', 
    legend=False              
)
axes[1].set_title('Tempo de Emprego (person_emp_length)', fontsize=12, pad=10)
axes[1].set_xlabel('Status do Empréstimo (0=Pago, 1=Inadimplente)', fontsize=10)
axes[1].set_ylabel('Anos de Emprego', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Boxplots de Controle Pós-Tratamento de Outliers', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()




# Justificativa Técnica para tratamento

**Justificativa para a Abordagem Combinada de Dados Ausentes:**  
Nesta etapa, optou-se por uma estratégia mista para o tratamento de dados nulos, avaliando o impacto de cada variável tanto do ponto de vista estatístico quanto do negócio:
1. **Remoção em `person_emp_length` (Tempo de Emprego):** O tempo de emprego é um indicador crucial para a análise de risco de crédito. A ausência desse dado pode sugerir inconsistência cadastral ou ocultação de informação (como desemprego). Como o volume de dados nulos nesta coluna representava um percentual muito baixo (aproximadamente 3% da base), a remoção completa dessas linhas limpa o dataset sem causar perda significativa de informação ou poder preditivo.
2. **Imputação por Mediana em `loan_int_rate` (Taxa de Juros):** Diferente da anterior, a taxa de juros possuía quase 10% de dados ausentes. Eliminar todas essas linhas causaria um descarte massivo de dados úteis (*data wastage*), reduzindo drasticamente a capacidade de aprendizado dos modelos. Optou-se pela **Mediana** por ser uma métrica de tendência central robusta, ideal para distribuições com forte assimetria e presença de *outliers* (como visto na EDA). A média distorceria o valor imputado para cima ou para baixo devido às taxas extremas, enquanto a mediana garante neutralidade estatística.

**Decisão de Tratamento de Outliers e Impacto nos Modelos:**  
Registros com dados impossíveis ou severamente discrepantes (idade superior ou igual a 80 anos e tempo de emprego incompatível com uma carreira profissional ativa) foram identificados nos gráficos da EDA e cirurgicamente **removidos**, garantindo que a massa de dados final represente clientes reais do mercado de crédito atual.

O impacto dessa higienização reflete diretamente na arquitetura dos algoritmos que serão testados:
* **Algoritmo KNN (K-Nearest Neighbors):** É **altamente vulnerável** a outliers. Como o KNN calcula a proximidade entre clientes usando fórmulas de distância geométrica no espaço multidimensional, a presença de valores absurdos de idade ou tempo de emprego distorceria a escala espacial inteira, aproximando incorretamente perfis totalmente diferentes.
* **Árvore de Decisão:** É **inerentemente robusta** a outliers, pois faz divisões por regras condicionais simples (ex: `taxa_juros > 12%`). O algoritmo não se importa com a magnitude do extremo, apenas com o ponto de quebra. Contudo, restringir a idade para menores de 80 anos e ajustar o tempo de emprego evita a criação de ramificações inúteis e folhas isoladas para comportamentos bizarros, diminuindo as chances de *overfitting* (sobreajuste).
